# First model: prepare the combined data

`combining_data.ipynb` already joins TMDB and IMDb, removes missing join IDs, and removes duplicate movie–director pairs. Start from its saved output instead of repeating the merge.

The combined file has multiple rows for co-directed movies. Group those directors before movie-level analysis. Only matched movies are available here; this does not restore the unmatched TMDB movies.


In [1]:
import pandas as pd

combined = pd.read_csv(
    "data/combined_dataset.csv",
    dtype={"imdb_id": "string", "nconst": "string", "collection_id": "Int64", "startYear": "Int64"},
)
print(f"Loaded {len(combined):,} movie–director rows")


Loaded 14,824 movie–director rows


## Keep one row per movie

Collect all IMDb directors for each movie, then attach that information to one copy of its movie characteristics.


In [2]:
def join_names(values):
    return "|".join(values.dropna().drop_duplicates())

directors = combined.groupby("tmdb_id", as_index=False).agg(
    imdb_director_ids=("nconst", join_names),
    imdb_directors=("director_imdb", join_names),
    imdb_director_count=("nconst", "nunique"),
)

movies = (
    combined.drop(columns=["nconst", "director_imdb"])
    .drop_duplicates("tmdb_id")
    .merge(directors, on="tmdb_id", validate="one_to_one", sort=False)
)
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
assert movies["tmdb_id"].is_unique
print(f"Ready for analysis: {len(movies):,} unique movies")
display(movies.head())


Ready for analysis: 13,276 unique movies


,tmdb_id,imdb_id,title_tmdb,release_date,budget,revenue,runtime,genres,original_language,popularity,...,vote_count,vote_average,collection_id,collection_name,us_certification,title_imdb,startYear,imdb_director_ids,imdb_directors,imdb_director_count
0,769,tt0099685,GoodFellas,1990-09-12,25000000,47072327,145,Drama|Crime,en,32.6399,...,14953,8.451,<NA>,NaN,R,GoodFellas,1990,nm0000217,Martin Scorsese,1
1,162,tt0099487,Edward Scissorhands,1990-12-07,20000000,86024005,105,Fantasy|Drama|Romance,en,17.6934,...,14152,7.719,<NA>,NaN,PG-13,Edward Scissorhands,1990,nm0000318,Tim Burton,1
2,771,tt0099785,Home Alone,1990-11-16,18000000,476684675,103,Comedy|Family,en,1.7507,...,12999,7.470,9888,Home Alone Collection,PG,Home Alone,1990,nm0001060,Chris Columbus,1
3,196,tt0099088,Back to the Future Part III,1990-05-25,40000000,245077583,119,Adventure|Comedy|Science Fiction,en,15.8103,...,11962,7.495,264,Back to the Future Collection,PG,Back to the Future Part III,1990,nm0000709,Robert Zemeckis,1
4,114,tt0100405,Pretty Woman,1990-03-23,14000000,463406268,120,Romance|Comedy,en,17.7921,...,9275,7.441,<NA>,NaN,R,Pretty Woman,1990,nm0005190,Garry Marshall,1


## Check what still needs cleaning

Combining did not resolve missing financial information. Zero budget makes revenue/budget undefined; zero revenue may mean unreported data. These counts are checks, not automatic deletion rules.


In [3]:
display(pd.DataFrame({
    "missing": movies[["budget", "revenue", "runtime", "release_date", "vote_average"]].isna().sum(),
    "zeros": movies[["budget", "revenue", "runtime", "release_date", "vote_average"]].eq(0).sum(),
}))
financial_mask = movies["budget"].gt(0) & movies["revenue"].gt(0)
print(f"Movies with positive budget and revenue: {financial_mask.sum():,}")


,missing,zeros
budget,0,5439
revenue,0,4112
runtime,0,8
release_date,0,0
vote_average,0,0


Movies with positive budget and revenue: 7,031


## Budget and revenue groups

Each unique movie belongs to one group below. The positive/positive group is the starting sample for calculating revenue divided by budget, subject to further validation. Zero values may indicate unreported data. No movies are removed.


In [4]:
budget_positive = movies["budget"].gt(0)
revenue_positive = movies["revenue"].gt(0)
budget_zero = movies["budget"].eq(0)
revenue_zero = movies["revenue"].eq(0)

financial_groups = pd.DataFrame([
    ["Positive", "Positive", (budget_positive & revenue_positive).sum()],
    ["Zero", "Positive", (budget_zero & revenue_positive).sum()],
    ["Positive", "Zero", (budget_positive & revenue_zero).sum()],
    ["Zero", "Zero", (budget_zero & revenue_zero).sum()],
], columns=["Budget", "Revenue", "Movies"])

# Keep missing or negative values separate if they appear in a future dataset.
other_count = len(movies) - financial_groups["Movies"].sum()
if other_count:
    financial_groups.loc[len(financial_groups)] = ["Missing / invalid", "Either field", other_count]
financial_groups["Percent of movies"] = financial_groups["Movies"].div(len(movies)).mul(100)
financial_mask = budget_positive & revenue_positive
financial_groups.loc[len(financial_groups)] = ["TOTAL", "All groups", len(movies), 100.0]

assert financial_groups.iloc[:-1]["Movies"].sum() == len(movies)
display(financial_groups.style.hide(axis="index").format({
    "Movies": "{:,.0f}", "Percent of movies": "{:.1f}%",
}))


Budget,Revenue,Movies,Percent of movies
Positive,Positive,"7,031",53.0%
Zero,Positive,"2,133",16.1%
Positive,Zero,806,6.1%
Zero,Zero,"3,306",24.9%
TOTAL,All groups,"13,276",100.0%


## New source fields

IMDb `startYear` is the release year reported by IMDb; keep it separate from TMDB’s full `release_date`. Collection fields describe franchise membership, and `us_certification` is a content rating, not the viewer-rating target. Blank collection fields usually mean no recorded collection; blank certification/year values remain unknown.


In [5]:
new_fields = ["startYear", "collection_id", "collection_name", "us_certification"]
display(pd.DataFrame({
    "type": movies[new_fields].dtypes.astype(str),
    "missing": movies[new_fields].isna().sum(),
    "missing_percent": movies[new_fields].isna().mean().mul(100).round(1),
}))
print("Movies with a recorded collection:", movies["collection_id"].notna().sum())
display(movies["us_certification"].fillna("Unknown").value_counts().rename_axis("US certification").to_frame("Movies"))
known_years = movies["startYear"].notna() & movies["release_date"].notna()
print("IMDb/TMDB release-year differences:", (
    movies.loc[known_years, "startYear"] != movies.loc[known_years, "release_date"].dt.year
).sum())


,type,missing,missing_percent
startYear,Int64,0,0.0
collection_id,Int64,10563,79.6
collection_name,str,10563,79.6
us_certification,str,2864,21.6


Movies with a recorded collection: 2713


,Movies
US certification,
R,5243
PG-13,2866
Unknown,2864
PG,1203
NR,831
G,218
NC-17,51


IMDb/TMDB release-year differences: 785


## Next steps

Use `movies` for movie-level analysis and use `combined` for director–movie relationships. Director-history features and model training are not implemented yet. Financial eligibility and bust/average/hit boundaries still need to be finalized. The source CSVs remain unchanged.
